# Faunari · Notebook 03 — De-dup, Safety Taxonomy & Leakage-safe Split

The **Phase-0 closer**: turns the raw multi-source download (`../data/raw/`) into a
clean, leakage-safe, labelled dataset ready for modeling. Everything here is framed by
the project's **asymmetric-error** objective — a venomous snake misread as safe can kill,
so the **binary danger label** (`venomous` / `non_venomous`) is the safety-critical target
and **recall on the venomous class** is what we optimize. Species is secondary/informational.

**What this notebook does (in order)**
1. **Unified master index** — one row per image: `path, source, species, venom_label,
   license, width, height`. Venom from folder structure (github/kaggle, hyphen/space
   normalized); iNat species from manifest, venom filled via the safety taxonomy.
2. **Species→venom danger mapping** — an explicit, expert-reviewable dict where the rule is
   **"medically dangerous to humans"**, NOT "produces any venom" (so rear-fanged harmless
   colubrids = `non_venomous`). Covers all 154 iNat species; uncertain ones are **flagged**.
   Persisted to `data/taxonomy/species_venom_map.csv`.
3. **Full cross-source de-duplication** — `imagehash.phash`, Hamming ≤ 6, on the **full**
   corpus. Cluster near-dups, assign `dup_cluster_id` to every row, keep one representative
   per cluster (higher resolution, then more permissive license).
4. **Leakage-safe, group-aware split** — 70/15/15, grouping unit = `dup_cluster_id` (near-dups
   never span splits), stratified by `venom_label`.
5. **Hard-case / look-alike test set** — a dedicated hold-out of known mimic confusions
   (rat snake vs cobra, wolf snake vs krait) + distant/low-res, kept separate and leak-free.
6. **Persist** — `data/processed/master_index.csv` + a dedup/split summary.

**Required libs** (all present in the `faunari` env): `pandas numpy pillow imagehash matplotlib tqdm`

In [1]:
# Imports + path resolution at the top so the whole pipeline is configurable in one place.
from __future__ import annotations

import warnings
from collections import Counter, defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Iterator

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import imagehash
from PIL import Image, ImageFile, UnidentifiedImageError

try:
    from tqdm.auto import tqdm
except ImportError:  # tqdm is optional; degrade to a no-op wrapper.
    def tqdm(it, **_):  # type: ignore
        return it

# Truncated JPEGs are common in scraped data; allow load but we still flag unreadable files.
ImageFile.LOAD_TRUNCATED_IMAGES = True
warnings.filterwarnings("ignore", category=UserWarning, module="PIL")

# Resolve repo root whether run from Trials/ or the project root (same rule as nb 01/02).
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "Trials" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
MANIFEST_DIR = DATA_DIR / "manifests"
TAXONOMY_DIR = DATA_DIR / "taxonomy"
PROCESSED_DIR = DATA_DIR / "processed"

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".gif", ".tif", ".tiff", ".webp"}
RANDOM_SEED = 42  # logged for reproducibility of every sample/split below.

for d in (TAXONOMY_DIR, PROCESSED_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f"Project root : {PROJECT_ROOT}")
print(f"Raw exists   : {RAW_DIR.exists()} | Manifest exists: {MANIFEST_DIR.exists()}")
print(f"Outputs      : {TAXONOMY_DIR}  ,  {PROCESSED_DIR}")

Project root : e:\Coding_Notes\Faunari
Raw exists   : True | Manifest exists: True
Outputs      : e:\Coding_Notes\Faunari\data\taxonomy  ,  e:\Coding_Notes\Faunari\data\processed


C:\Users\Dhruv\miniforge3\envs\faunari\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1 · Build the unified master index

One row per image with provenance + on-disk facts. Venom label comes from the **folder
structure** for the two binary sources, with the two on-disk spellings normalized to a
single canonical vocabulary:

- GitHub: `Venomous` / `Non-Venomous` (hyphen), species nested one level below.
- Kaggle: `Snake Images/{train,test}/{Venomous, Non Venomous}` (space), **no species**.
- iNaturalist: flat folder, **species from the manifest**, venom filled in §2 via the taxonomy.

Unreadable files are **counted and reported, never silently dropped** — in a safety
pipeline a silently missing image is a silently missing data point.

In [2]:
def normalize_venom_label(folder: str) -> str | None:
    """Map any source's class-folder name to {'venomous','non_venomous'} or None if not a label.

    Handles the two on-disk spellings found in EDA: 'Non-Venomous' (hyphen, GitHub) and
    'Non Venomous' (space, Kaggle) -> both canonicalize to 'non_venomous'.
    """
    key = folder.strip().lower().replace("-", " ")
    if key == "venomous":
        return "venomous"
    if key in {"non venomous", "nonvenomous"}:
        return "non_venomous"
    return None


def iter_image_paths(root: Path) -> Iterator[Path]:
    """Yield every image-extension file under root, skipping VCS dirs (e.g. the GitHub .git)."""
    if not root.exists():
        return
    for p in root.rglob("*"):
        if ".git" in p.parts:
            continue
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS:
            yield p

In [3]:
def labels_from_parts(source: str, rel_parts: tuple[str, ...]) -> tuple[str | None, str | None]:
    """Infer (venom_label, species_folder) from path parts for the folder-labelled sources.

    GitHub: <Venom>/<Species>/img            Kaggle: 'Snake Images'/<split>/<Venom>/img
    iNaturalist carries no folder labels here -> (None, None); species comes from the manifest.
    """
    venom: str | None = None
    species: str | None = None
    for i, part in enumerate(rel_parts):
        lab = normalize_venom_label(part)
        if lab is not None:
            venom = lab
            # GitHub nests a species folder directly under the venom folder; Kaggle does not.
            if source == "github_indian_snakes" and i + 1 < len(rel_parts) - 1:
                species = rel_parts[i + 1]
            break
    return venom, species


@dataclass
class ImageRecord:
    """One probed image: provenance + on-disk facts (single responsibility for downstream joins)."""
    source: str
    path: str
    species: str | None       # scientific (iNat, via manifest) OR common-name folder (GitHub)
    venom_label: str | None   # from folder structure where it exists; iNat filled via taxonomy
    license: str | None
    width: int | None
    height: int | None
    corrupt: bool
    error: str | None


def probe_image(path: Path, source: str, species: str | None,
                venom_label: str | None, license_: str | None) -> ImageRecord:
    """Read one image header for w/h, flagging unreadable files instead of raising (keep scan going)."""
    try:
        with Image.open(path) as im:
            im.verify()                       # cheap integrity check, no full decode
        with Image.open(path) as im:
            w, h = im.size
        return ImageRecord(source, str(path), species, venom_label, license_, w, h,
                           corrupt=False, error=None)
    except (UnidentifiedImageError, OSError, ValueError) as exc:
        return ImageRecord(source, str(path), species, venom_label, license_, None, None,
                           corrupt=True, error=str(exc))

In [4]:
def load_inat_lookup() -> dict[str, dict[str, str | None]]:
    """Map image filename -> {species, license} from the iNaturalist manifest (defensive load).

    Keyed by filename (the record_id.jpg) rather than the manifest's absolute local_path, so the
    join is robust whether the corpus walk yields relative or absolute paths.
    """
    path = MANIFEST_DIR / "inaturalist_manifest.csv"
    if not path.exists():
        print(f"[skip] iNat manifest not found at {path}")
        return {}
    df = pd.read_csv(path, dtype=str)
    lookup: dict[str, dict[str, str | None]] = {}
    for _, r in df.iterrows():
        lp = r.get("local_path")
        if isinstance(lp, str) and lp.strip():
            lic = r.get("license")
            lookup[Path(lp).name] = {
                "species": (r.get("scientific_name") or None),
                "license": (lic if isinstance(lic, str) and lic.strip() else None),
            }
    print(f"[ok]   iNat manifest: {len(lookup)} filename->meta entries")
    return lookup


INAT_LOOKUP = load_inat_lookup()

[ok]   iNat manifest: 2000 filename->meta entries


In [5]:
def build_master_records() -> list[ImageRecord]:
    """Walk every source once, attaching folder/manifest-derived labels (no de-dup yet)."""
    source_dirs = {
        "inaturalist":          RAW_DIR / "inaturalist",
        "github_indian_snakes": RAW_DIR / "github_indian_snakes",
        "kaggle_india":         RAW_DIR / "kaggle_india",
    }
    records: list[ImageRecord] = []
    for source, root in source_dirs.items():
        if not root.exists():
            print(f"[skip] {source}: directory not found at {root}")
            continue
        n = 0
        for p in iter_image_paths(root):
            rel_parts = p.relative_to(root).parts
            venom, species = labels_from_parts(source, rel_parts)
            license_ = None
            if source == "inaturalist":
                meta = INAT_LOOKUP.get(p.name, {})
                species = meta.get("species")          # scientific name from manifest
                license_ = meta.get("license")
            records.append(probe_image(p, source, species, venom, license_))
            n += 1
        print(f"[ok]   {source}: {n} image files probed")
    return records


master = pd.DataFrame([r.__dict__ for r in build_master_records()])
n_corrupt = int(master["corrupt"].sum())
print(f"\nTotal probed: {len(master)} | corrupt/unreadable: {n_corrupt}")
# Drop corrupt rows from the modeling index but keep the count reported above (honest totals).
master = master[~master["corrupt"]].drop(columns=["corrupt", "error"]).reset_index(drop=True)
print(f"Usable images in master index: {len(master)}")
master.head()

[ok]   inaturalist: 2000 image files probed


[ok]   github_indian_snakes: 1779 image files probed


[ok]   kaggle_india: 2044 image files probed

Total probed: 5823 | corrupt/unreadable: 0
Usable images in master index: 5823


,source,path,species,venom_label,license,width,height
0,inaturalist,e:\Coding_Notes\Faunari\data\raw\inaturalist\3...,Daboia russelii,NaN,cc-by-nc,500,478
1,inaturalist,e:\Coding_Notes\Faunari\data\raw\inaturalist\3...,Oligodon taeniolatus,NaN,cc-by-nc,281,500
2,inaturalist,e:\Coding_Notes\Faunari\data\raw\inaturalist\3...,Ptyas mucosa,NaN,cc-by-nc,500,281
3,inaturalist,e:\Coding_Notes\Faunari\data\raw\inaturalist\3...,Ovophis monticola,NaN,NaN,500,311
4,inaturalist,e:\Coding_Notes\Faunari\data\raw\inaturalist\3...,Daboia russelii,NaN,cc-by-nc,500,311


## 2 · Species → venom **danger** mapping (the safety taxonomy)

The crux of the safety design. The rule is **"medically dangerous to humans"**, *not*
"produces any venom". Many Indian colubrids are technically rear-fanged/mildly venomous
but cause **no medically significant envenomation** in humans — calling them `venomous`
would (a) train the model on a wrong boundary and (b) flood the dangerous class with
harmless look-alikes. So:

- **`non_venomous`** (not medically dangerous): harmless & rear-fanged colubrids
  (`Ptyas`, `Lycodon`, `Oligodon`, `Coelognathus`, `Elaphe`, `Dendrelaphis`, `Boiga`,
  `Ahaetulla`, `Fowlea`, `Amphiesma`, `Atretium`, `Chrysopelea`, `Psammophis`, …),
  pythons/boas (`Python`, `Malayopython`, `Eryx`, `Xenopeltis`), blind/shield/water snakes.
- **`venomous`** (medically dangerous): elapids (`Naja`, `Bungarus`, `Ophiophagus`,
  `Calliophis`, `Sinomicrurus`), true vipers (`Daboia`, `Echis`), pit vipers
  (`Craspedocephalus`, `Trimeresurus`, `Hypnale`, `Gloydius`, `Ovophis`).

The mapping is **genus-driven** (robust to the long tail of single-image species) with
explicit per-species overrides where genus alone is ambiguous. Every one of the 154 iNat
species is covered, and genuine edge cases are **flagged for human (herpetologist) review**
rather than guessed silently.

In [6]:
# Genus-level danger rule, keyed by the FIRST token of the scientific name.
# Rationale: danger is overwhelmingly a clade property in Indian snakes, and a genus rule
# generalizes to the long tail of 1-image species without per-species guesswork.

VENOMOUS_GENERA = {
    # Elapids (front-fanged, medically dangerous)
    "Naja", "Bungarus", "Ophiophagus", "Calliophis", "Sinomicrurus",
    # True vipers
    "Daboia", "Echis",
    # Pit vipers
    "Craspedocephalus", "Trimeresurus", "Hypnale", "Gloydius", "Ovophis",
}

NON_VENOMOUS_GENERA = {
    # Harmless / rat / wolf / kukri / trinket / racer / bronzeback colubrids
    "Ptyas", "Lycodon", "Oligodon", "Coelognathus", "Elaphe", "Platyceps",
    "Spalerosophis", "Sibynophis", "Oreocryptophis", "Gonyosoma", "Dendrelaphis",
    # Rear-fanged but NOT medically dangerous to humans (cat snakes, vine snakes, etc.)
    "Boiga", "Ahaetulla", "Chrysopelea", "Psammodynastes", "Psammophis",
    # Keelbacks / water & marsh colubrids (mild or no medical significance)
    "Fowlea", "Amphiesma", "Atretium", "Herpetoreas", "Xenochrophis",
    "Pseudoxenodon", "Cerberus", "Enhydris", "Hypsiscopus", "Dieurostus",
    "Rhabdops", "Pareas", "Sahyadriophis",
    # Pythons & boas (constrictors, non-venomous)
    "Python", "Malayopython", "Eryx", "Xenopeltis",
    # Blind / worm / shield-tail / file snakes (non-venomous)
    "Indotyphlops", "Grypotyphlops", "Argyrophis", "Pseudoindotyphlops",
    "Uropeltis", "Rhinophis", "Melanophidium", "Plectrurus", "Platyplectrurus",
    "Xylophis", "Acrochordus",
}

# Per-species overrides / clarifications where the genus is mixed or warrants a note.
SPECIES_OVERRIDES: dict[str, str] = {
    # Rhabdophis: Asian keelbacks. R. plumbicolor/helleri etc. are nuchal-gland/rear-fanged
    # and bites are not part of India's medically-important burden -> non_venomous here.
    "Rhabdophis plumbicolor": "non_venomous",
    "Rhabdophis plumbicolor plumbicolor": "non_venomous",
    "Rhabdophis helleri": "non_venomous",
    "Rhabdophis leonardi": "non_venomous",
    "Rhabdophis himalayanus": "non_venomous",
}

# Species/genera we are NOT confident classifying as medically dangerous -> flag for an
# expert herpetologist, defaulting to the SAFER classification only after review.
# (Rationale: asymmetric error - we must not silently call something non_venomous if unsure.)
REVIEW_FLAG: dict[str, str] = {
    # Genus Rhabdophis has documented severe envenomations elsewhere (R. tigrinus/subminiatus
    # in E. Asia). The Indian spp. here are not in the medical-importance list, but the genus
    # merits a human sign-off given the asymmetric cost.
    "Rhabdophis plumbicolor": "rear-fanged keelback; genus has dangerous members abroad - confirm Indian medical relevance",
    "Rhabdophis plumbicolor plumbicolor": "see Rhabdophis plumbicolor",
    "Rhabdophis helleri": "Rhabdophis - confirm not medically significant in India",
    "Rhabdophis leonardi": "Rhabdophis - confirm not medically significant in India",
    "Rhabdophis himalayanus": "Rhabdophis - confirm not medically significant in India",
}

In [7]:
def classify_species_danger(scientific_name: str) -> tuple[str, bool, str]:
    """Return (venom_label, needs_review, reason) for one scientific name using the safety rule.

    Order: explicit species override -> genus rule -> unknown (flagged). 'needs_review' marks
    genuine edge cases for a herpetologist; we never silently guess a dangerous species safe.
    """
    name = (scientific_name or "").strip()
    if not name:
        return ("non_venomous", True, "empty/missing scientific name")
    genus = name.split()[0]

    review_note = REVIEW_FLAG.get(name, "")
    if name in SPECIES_OVERRIDES:
        return (SPECIES_OVERRIDES[name], bool(review_note), review_note or "species override")
    if genus in VENOMOUS_GENERA:
        return ("venomous", bool(review_note), review_note or f"genus {genus} = medically dangerous")
    if genus in NON_VENOMOUS_GENERA:
        return ("non_venomous", bool(review_note), review_note or f"genus {genus} = not medically dangerous")
    # Unknown genus: default to the SAFER side for labeling but FLAG it loudly for review.
    return ("non_venomous", True, f"UNKNOWN genus '{genus}' - needs expert review")


def build_species_venom_map(species_names: list[str]) -> pd.DataFrame:
    """One row per unique iNat species with label, review flag, reason, and genus (auditable)."""
    rows = []
    for name in sorted(set(s for s in species_names if isinstance(s, str) and s.strip())):
        label, review, reason = classify_species_danger(name)
        rows.append({"scientific_name": name, "genus": name.split()[0],
                     "venom_label": label, "needs_review": review, "reason": reason})
    return pd.DataFrame(rows)


inat_species = master.loc[master["source"] == "inaturalist", "species"].dropna().tolist()
species_map = build_species_venom_map(inat_species)
print(f"Species covered: {len(species_map)} (expected 154)")
print(species_map["venom_label"].value_counts().to_string())
flagged = species_map[species_map["needs_review"]]
print(f"\nFlagged for human review: {len(flagged)}")
print(flagged[["scientific_name", "venom_label", "reason"]].to_string(index=False) if len(flagged)
      else "  (none)")

Species covered: 154 (expected 154)
venom_label
non_venomous    122
venomous         32

Flagged for human review: 5
                   scientific_name  venom_label                                                                                      reason
                Rhabdophis helleri non_venomous                                     Rhabdophis - confirm not medically significant in India
            Rhabdophis himalayanus non_venomous                                     Rhabdophis - confirm not medically significant in India
               Rhabdophis leonardi non_venomous                                     Rhabdophis - confirm not medically significant in India
            Rhabdophis plumbicolor non_venomous rear-fanged keelback; genus has dangerous members abroad - confirm Indian medical relevance
Rhabdophis plumbicolor plumbicolor non_venomous                                                                  see Rhabdophis plumbicolor


In [8]:
# Persist the safety taxonomy so it is expert-reviewable and reusable by every later notebook.
SPECIES_MAP_PATH = TAXONOMY_DIR / "species_venom_map.csv"
species_map.to_csv(SPECIES_MAP_PATH, index=False)
print(f"Wrote safety taxonomy -> {SPECIES_MAP_PATH}  ({len(species_map)} species)")

# Sanity-check the big four are labelled venomous (a hard requirement of the objective).
BIG_FOUR = ["Naja naja", "Bungarus caeruleus", "Daboia russelii", "Echis carinatus"]
bf = species_map[species_map["scientific_name"].isin(BIG_FOUR)][["scientific_name", "venom_label"]]
print("\nBig-four label check:")
print(bf.to_string(index=False))
assert (bf["venom_label"] == "venomous").all(), "Big-four must be venomous!"
print("OK: all big-four labelled venomous.")

Wrote safety taxonomy -> e:\Coding_Notes\Faunari\data\taxonomy\species_venom_map.csv  (154 species)

Big-four label check:
   scientific_name venom_label
Bungarus caeruleus    venomous
   Daboia russelii    venomous
   Echis carinatus    venomous
         Naja naja    venomous
OK: all big-four labelled venomous.


In [9]:
def fill_inat_venom(df: pd.DataFrame, smap: pd.DataFrame) -> pd.DataFrame:
    """Fill venom_label for iNat rows from the species taxonomy; leave folder-labelled rows intact."""
    label_by_species = dict(zip(smap["scientific_name"], smap["venom_label"]))
    out = df.copy()
    is_inat = out["source"] == "inaturalist"
    out.loc[is_inat, "venom_label"] = out.loc[is_inat, "species"].map(label_by_species)
    return out


master = fill_inat_venom(master, species_map)
missing_lbl = int(master["venom_label"].isna().sum())
print(f"Rows missing venom_label after fill: {missing_lbl}")
if missing_lbl:
    print(master[master["venom_label"].isna()]["source"].value_counts().to_string())
print("\nVenom balance across the FULL master index:")
print(master["venom_label"].value_counts(dropna=False).to_string())

Rows missing venom_label after fill: 0

Venom balance across the FULL master index:
venom_label
non_venomous    2959
venomous        2864


## 3 · Full cross-source de-duplication (`phash`, Hamming ≤ 6)

EDA found ~164 cross-source **github↔kaggle** near-duplicate pairs — the single biggest
leakage hotspot. Here we run the perceptual hash on the **full** corpus (no sampling),
build connected components of near-duplicates (transitive closure: if A≈B and B≈C they
share a cluster), and assign a `dup_cluster_id` to **every** image (singletons get their
own id). We keep one representative per cluster — **higher resolution first, then more
permissive license** — but retain cluster ids on all rows so the split can group by them.

In [10]:
def compute_phash(path: str) -> imagehash.ImageHash | None:
    """Perceptual hash of one image; None on unreadable files (already counted in §1)."""
    try:
        with Image.open(path) as im:
            return imagehash.phash(im.convert("RGB"))
    except (UnidentifiedImageError, OSError, ValueError):
        return None


PHASH_THRESHOLD = 6  # Hamming <=6 on a 64-bit phash ~ visually near-identical (matches nb 02).

phashes: list[imagehash.ImageHash | None] = []
n_hash_fail = 0
for p in tqdm(master["path"].tolist(), desc="phash (full corpus)"):
    h = compute_phash(p)
    if h is None:
        n_hash_fail += 1
    phashes.append(h)
master["_phash"] = phashes
print(f"Hashed {len(master)} images | hash failures: {n_hash_fail}")

phash (full corpus):   0%|          | 0/5823 [00:00<?, ?it/s]

phash (full corpus):   0%|          | 1/5823 [00:03<5:44:54,  3.55s/it]

phash (full corpus):   0%|          | 12/5823 [00:03<21:30,  4.50it/s] 

phash (full corpus):   0%|          | 21/5823 [00:03<10:52,  8.89it/s]

phash (full corpus):   0%|          | 29/5823 [00:03<07:02, 13.70it/s]

...

phash (full corpus):  99%|█████████▉| 5793/5823 [01:57<00:00, 122.39it/s]

phash (full corpus): 100%|█████████▉| 5806/5823 [01:57<00:00, 118.93it/s]

phash (full corpus): 100%|█████████▉| 5820/5823 [01:57<00:00, 122.18it/s]

phash (full corpus): 100%|██████████| 5823/5823 [01:58<00:00, 49.34it/s] 

Hashed 5823 images | hash failures: 0


In [11]:
class UnionFind:
    """Disjoint-set for grouping near-duplicates into connected components (near-linear)."""

    def __init__(self, n: int) -> None:
        self.parent = list(range(n))
        self.rank = [0] * n

    def find(self, x: int) -> int:
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]  # path halving
            x = self.parent[x]
        return x

    def union(self, a: int, b: int) -> None:
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return
        if self.rank[ra] < self.rank[rb]:
            ra, rb = rb, ra
        self.parent[rb] = ra
        if self.rank[ra] == self.rank[rb]:
            self.rank[ra] += 1


def cluster_near_duplicates(hashes: list, threshold: int) -> tuple[np.ndarray, list[tuple[int, int, int]]]:
    """Assign each image a dup_cluster_id via union-find over near-duplicate pairs.

    Blocks by hash prefix (cheap 16-bit key) so we only do the O(k^2) Hamming compare inside
    small buckets -> near-O(n) overall instead of all-pairs. Returns (cluster_ids, pair_list).
    """
    n = len(hashes)
    uf = UnionFind(n)
    buckets: dict[str, list[int]] = defaultdict(list)
    for i, h in enumerate(hashes):
        if h is not None:
            buckets[str(h)[:4]].append(i)

    pairs: list[tuple[int, int, int]] = []  # (idx_a, idx_b, distance) for reporting
    for idxs in buckets.values():
        for a in range(len(idxs)):
            for b in range(a + 1, len(idxs)):
                i, j = idxs[a], idxs[b]
                dist = hashes[i] - hashes[j]
                if dist <= threshold:
                    uf.union(i, j)
                    pairs.append((i, j, dist))

    # Compact the root labels into contiguous 0..K-1 cluster ids.
    roots = [uf.find(i) for i in range(n)]
    remap = {r: cid for cid, r in enumerate(sorted(set(roots)))}
    cluster_ids = np.array([remap[r] for r in roots], dtype=int)
    return cluster_ids, pairs


master["dup_cluster_id"], dup_pairs = cluster_near_duplicates(master["_phash"].tolist(), PHASH_THRESHOLD)
n_clusters = master["dup_cluster_id"].nunique()
print(f"Clusters: {n_clusters} for {len(master)} images "
      f"({len(master) - n_clusters} images are near-duplicates of another).")

Clusters: 4176 for 5823 images (1647 images are near-duplicates of another).


In [12]:
def summarize_duplicates(df: pd.DataFrame, pairs: list[tuple[int, int, int]]) -> None:
    """Report near-duplicate pair counts within vs across sources (leakage diagnostics)."""
    src = df["source"].to_numpy()
    within = sum(1 for i, j, _ in pairs if src[i] == src[j])
    cross = len(pairs) - within
    print(f"Near-duplicate pairs (Hamming <= {PHASH_THRESHOLD}): {len(pairs)} total")
    print(f"  within-source: {within}  |  CROSS-source (leakage risk): {cross}")
    if cross:
        cc = Counter(tuple(sorted((src[i], src[j]))) for i, j, _ in pairs if src[i] != src[j])
        print("\n  Cross-source pairs by source pair:")
        for (sa, sb), n in cc.most_common():
            print(f"    {sa} <-> {sb}: {n}")

    # Multi-image clusters: how big are the dup groups, and which sources do they span?
    sizes = df.groupby("dup_cluster_id").size()
    multi = sizes[sizes > 1]
    print(f"\n  Multi-image clusters: {len(multi)} (covering {int(multi.sum())} images)")
    if len(multi):
        spanning = (df[df["dup_cluster_id"].isin(multi.index)]
                    .groupby("dup_cluster_id")["source"].nunique())
        print(f"  Clusters spanning >1 source: {int((spanning > 1).sum())}")
        print(f"  Largest cluster size: {int(sizes.max())}")


summarize_duplicates(master, dup_pairs)

Near-duplicate pairs (Hamming <= 6): 1711 total
  within-source: 72  |  CROSS-source (leakage risk): 1639

  Cross-source pairs by source pair:
    github_indian_snakes <-> kaggle_india: 1639

  Multi-image clusters: 1595 (covering 3242 images)
  Clusters spanning >1 source: 1583
  Largest cluster size: 5


In [13]:
# Choose ONE representative per cluster: prefer higher resolution, then a more permissive license.
LICENSE_RANK = {  # lower = more permissive / more release-safe (used only as a tie-breaker)
    "cc0": 0, "cc-by": 1, "cc-by-sa": 2, "cc-by-nd": 3,
    "cc-by-nc": 4, "cc-by-nc-sa": 5, "cc-by-nc-nd": 6,
}


def pick_representatives(df: pd.DataFrame) -> pd.DataFrame:
    """Flag is_representative=True for one row per dup_cluster_id (max pixels, then best license)."""
    out = df.copy()
    out["_pixels"] = (out["width"].fillna(0) * out["height"].fillna(0)).astype(float)
    out["_lic_rank"] = out["license"].map(LICENSE_RANK).fillna(99).astype(int)
    # Sort so the preferred representative is first within each cluster, then take the head.
    ordered = out.sort_values(["dup_cluster_id", "_pixels", "_lic_rank"],
                              ascending=[True, False, True])
    rep_idx = ordered.groupby("dup_cluster_id", sort=False).head(1).index
    out["is_representative"] = out.index.isin(rep_idx)
    return out.drop(columns=["_pixels", "_lic_rank"])


master = pick_representatives(master)
n_rep = int(master["is_representative"].sum())
print(f"Representatives (deduped corpus size): {n_rep}")
print(f"Duplicates suppressed (retained but not representative): {len(master) - n_rep}")
print("\nVenom balance among representatives:")
print(master.loc[master["is_representative"], "venom_label"].value_counts(dropna=False).to_string())

Representatives (deduped corpus size): 4176
Duplicates suppressed (retained but not representative): 1647

Venom balance among representatives:
venom_label
non_venomous    2398
venomous        1778


## 4 · Leakage-safe, group-aware split (70 / 15 / 15)

The grouping unit is `dup_cluster_id`, so **near-duplicates can never span splits**.
We split over **representatives only** (one row per cluster), stratify by `venom_label`
so the dangerous class is proportionally present in every split, then propagate each
cluster's split assignment back to all its member rows. Splitting at the cluster level is
the only way to make reported venomous-recall trustworthy.

In [14]:
def group_stratified_split(reps: pd.DataFrame, seed: int,
                           frac_train: float = 0.70, frac_val: float = 0.15
                           ) -> dict[int, str]:
    """Assign each cluster (one rep row each) to train/val/test, stratified by venom_label.

    Operates on representatives (already 1 row per cluster) so the unit is the cluster.
    Greedy per-stratum assignment to the target fractions; deterministic under `seed`.
    Returns {dup_cluster_id: split}.
    """
    rng = np.random.default_rng(seed)
    assignment: dict[int, str] = {}
    for label, grp in reps.groupby("venom_label", dropna=False):
        cids = grp["dup_cluster_id"].to_numpy().copy()  # copy: groupby views are read-only
        rng.shuffle(cids)
        n = len(cids)
        n_train = int(round(n * frac_train))
        n_val = int(round(n * frac_val))
        for cid in cids[:n_train]:
            assignment[int(cid)] = "train"
        for cid in cids[n_train:n_train + n_val]:
            assignment[int(cid)] = "val"
        for cid in cids[n_train + n_val:]:
            assignment[int(cid)] = "test"
    return assignment


reps = master[master["is_representative"]]
split_by_cluster = group_stratified_split(reps, seed=RANDOM_SEED)
master["split"] = master["dup_cluster_id"].map(split_by_cluster)
assert master["split"].notna().all(), "Every image must receive a split."
print("Split assigned to all images via their cluster (no near-dup spans splits).")

Split assigned to all images via their cluster (no near-dup spans splits).


In [15]:
def report_split(df: pd.DataFrame, level: str) -> None:
    """Print per-split image counts and venom balance, plus a no-cluster-leak assertion."""
    print(f"=== {level} ===")
    tab = pd.crosstab(df["split"], df["venom_label"], margins=True)
    print(tab.to_string())
    frac = (df.groupby("split")["venom_label"]
            .apply(lambda s: (s == "venomous").mean()).round(3))
    print("\nVenomous fraction per split:")
    print(frac.to_string())


report_split(master, "All images (post-dedup, with suppressed dups kept in their cluster's split)")

# Hard guarantee: no dup_cluster_id appears in more than one split.
leak = master.groupby("dup_cluster_id")["split"].nunique()
n_leak = int((leak > 1).sum())
print(f"\nClusters spanning >1 split (MUST be 0): {n_leak}")
assert n_leak == 0, "Leakage: a dup cluster spans multiple splits!"

=== All images (post-dedup, with suppressed dups kept in their cluster's split) ===
venom_label  non_venomous  venomous   All
split                                    
test                  437       427   864
train                2077      2016  4093
val                   445       421   866
All                  2959      2864  5823



Venomous fraction per split:
split
test     0.494
train    0.493
val      0.486

Clusters spanning >1 split (MUST be 0): 0


## 5 · Hard-case / look-alike test set (the recall stress test)

A random test split flatters the model: it under-represents the **exact confusions that
get people killed**. We carve a *dedicated* hold-out of known mimic pairs and hard images:

- **Rat snake (`Ptyas mucosa`, non-venomous) vs cobra (`Naja naja`, venomous)** — the
  classic cobra mimic.
- **Wolf snake (`Lycodon aulicus`, non-venomous) vs krait (`Bungarus caeruleus`, venomous)**
  — the classic krait mimic, and the most dangerous mistake (kraits bite at night).
- A sample of **distant / low-resolution** images (small shorter-side) to test robustness.

This set is kept **separate** from the random test split and, because selection is at the
**cluster** level, its clusters are removed from train/val so nothing leaks in.

In [16]:
# Mimic pairs to over-sample into the hard set, addressed by scientific name (iNat) where possible.
HARDCASE_SPECIES = {
    "Ptyas mucosa":        "rat snake (non-venom) - cobra mimic",
    "Naja naja":           "spectacled cobra (venom)",
    "Lycodon aulicus":     "wolf snake (non-venom) - krait mimic",
    "Bungarus caeruleus":  "common krait (venom)",
}
LOWRES_SHORT_SIDE = 256  # 'distant/low-res': shorter side below a transfer-learning-friendly size.
HARD_PER_GROUP = 40      # cap mimic images per species so the set stays balanced & small.
HARD_LOWRES_N = 60       # how many distant/low-res images to add.


def select_hardcase_clusters(df: pd.DataFrame, seed: int) -> set[int]:
    """Pick dup_cluster_ids for the hard-case set: mimic species + a sample of low-res images.

    Selection is at the CLUSTER level so removing them from train/val cannot leak a near-dup.
    """
    rng = np.random.default_rng(seed)
    chosen: set[int] = set()

    # 1) Mimic species: take up to HARD_PER_GROUP clusters per target species.
    for sp in HARDCASE_SPECIES:
        cids = df.loc[df["species"] == sp, "dup_cluster_id"].dropna().unique().copy()
        rng.shuffle(cids)
        chosen.update(int(c) for c in cids[:HARD_PER_GROUP])

    # 2) Distant / low-res images: sample clusters whose representative is small.
    short_side = df[["width", "height"]].min(axis=1)
    lowres = df.loc[short_side < LOWRES_SHORT_SIDE, "dup_cluster_id"].dropna().unique()
    rng.shuffle(lowres)
    chosen.update(int(c) for c in lowres[:HARD_LOWRES_N])
    return chosen


hard_clusters = select_hardcase_clusters(master, seed=RANDOM_SEED)
# Reassign every image in a hard cluster to the dedicated 'hardcase' split (removes from train/val/test).
master.loc[master["dup_cluster_id"].isin(hard_clusters), "split"] = "hardcase"
print(f"Hard-case clusters: {len(hard_clusters)}")

Hard-case clusters: 215


In [17]:
# Re-verify integrity after carving the hard set, and report its composition.
leak = master.groupby("dup_cluster_id")["split"].nunique()
assert int((leak > 1).sum()) == 0, "Leakage after hard-case carve!"

hard = master[master["split"] == "hardcase"]
print(f"Hard-case set size (images): {len(hard)}")
print("\nHard-case venom balance:")
print(hard["venom_label"].value_counts(dropna=False).to_string())
print("\nHard-case species breakdown (mimic targets + low-res):")
print(hard["species"].fillna("(no species / kaggle)").value_counts().head(12).to_string())

report_split(master, "Final splits (train/val/test/hardcase) - ALL images")

Hard-case set size (images): 239

Hard-case venom balance:
venom_label
venomous        123
non_venomous    116

Hard-case species breakdown (mimic targets + low-res):
species
Ptyas mucosa                    46
Naja naja                       42
Bungarus caeruleus              41
Lycodon aulicus                 40
(no species / kaggle)           24
Russell's Viper                  5
Spectacled Cobra                 5
Common Krait                     3
Common Rat Snake                 3
Daboia russelii                  2
Craspedocephalus malabaricus     2
Craspedocephalus gramineus       2
=== Final splits (train/val/test/hardcase) - ALL images ===


venom_label  non_venomous  venomous   All
split                                    
hardcase              116       123   239
test                  424       409   833
train                1992      1931  3923
val                   427       401   828
All                  2959      2864  5823

Venomous fraction per split:
split
hardcase    0.515
test        0.491
train       0.492
val         0.484


## 6 · Persist outputs

- `data/processed/master_index.csv` — every usable image with `path, source, species,
  venom_label, license, width, height, dup_cluster_id, is_representative, split`.
- `data/processed/dedup_split_summary.txt` — a short, human-readable summary of the dedup
  numbers, split sizes, venom balance, and hard-case composition.
- `data/taxonomy/species_venom_map.csv` — already written in §2.

In [18]:
def write_master_index(df: pd.DataFrame) -> Path:
    """Persist the final per-image index with stable, modeling-ready columns."""
    cols = ["path", "source", "species", "venom_label", "license",
            "width", "height", "dup_cluster_id", "is_representative", "split"]
    out_path = PROCESSED_DIR / "master_index.csv"
    df[cols].to_csv(out_path, index=False)
    return out_path


def write_summary(df: pd.DataFrame, pairs: list[tuple[int, int, int]],
                  n_species: int, n_flagged: int) -> Path:
    """Write a concise dedup/split summary for the architect and reviewers."""
    lines: list[str] = []
    a = lines.append
    a("Faunari · Phase-0 dedup & split summary")
    a("=" * 44)
    a(f"Total usable images          : {len(df)}")
    a(f"Dup clusters (deduped size)  : {df['dup_cluster_id'].nunique()}")
    a(f"Representatives              : {int(df['is_representative'].sum())}")
    a(f"Near-duplicate pairs (<= {PHASH_THRESHOLD}) : {len(pairs)}")
    src = df["source"].to_numpy()
    cross = sum(1 for i, j, _ in pairs if src[i] != src[j])
    a(f"  cross-source pairs         : {cross}")
    a(f"Species in taxonomy          : {n_species} (flagged for review: {n_flagged})")
    a("")
    a("Per-split image counts:")
    a(df["split"].value_counts().to_string())
    a("")
    a("Venomous fraction per split:")
    frac = df.groupby("split")["venom_label"].apply(lambda s: (s == "venomous").mean()).round(3)
    a(frac.to_string())
    a("")
    a("Hard-case set:")
    hard = df[df["split"] == "hardcase"]
    a(f"  size: {len(hard)}  | venom balance: "
      f"{hard['venom_label'].value_counts(dropna=False).to_dict()}")
    out_path = PROCESSED_DIR / "dedup_split_summary.txt"
    out_path.write_text("\n".join(lines), encoding="utf-8")
    return out_path


index_path = write_master_index(master)
summary_path = write_summary(master, dup_pairs, len(species_map), int(species_map["needs_review"].sum()))
print(f"Wrote master index    -> {index_path}")
print(f"Wrote summary         -> {summary_path}")
print(f"Wrote safety taxonomy -> {SPECIES_MAP_PATH}")
print("\n--- summary ---")
print(summary_path.read_text(encoding="utf-8"))

Wrote master index    -> e:\Coding_Notes\Faunari\data\processed\master_index.csv


Wrote summary         -> e:\Coding_Notes\Faunari\data\processed\dedup_split_summary.txt
Wrote safety taxonomy -> e:\Coding_Notes\Faunari\data\taxonomy\species_venom_map.csv

--- summary ---
Faunari · Phase-0 dedup & split summary
Total usable images          : 5823
Dup clusters (deduped size)  : 4176
Representatives              : 4176
Near-duplicate pairs (<= 6) : 1711
  cross-source pairs         : 1639
Species in taxonomy          : 154 (flagged for review: 5)

Per-split image counts:
split
train       3923
test         833
val          828
hardcase     239

Venomous fraction per split:
split
hardcase    0.515
test        0.491
train       0.492
val         0.484

Hard-case set:
  size: 239  | venom balance: {'venomous': 123, 'non_venomous': 116}


## 7 · Findings & implications for modeling

> Numbers above are the source of truth; the points below are what actually changes the plan.

1. **Cross-source github↔kaggle duplication is the leakage hotspot — now neutralized.**
   Clustering by `dup_cluster_id` and splitting at the cluster level guarantees no near-dup
   spans train/val/test. Any future loader **must** join on `master_index.csv` and respect
   `split`; re-deriving splits from folders would silently re-introduce the leak.

2. **The danger label is taxonomy-driven, not "has venom".** Rear-fanged harmless colubrids
   (rat snake, wolf snakes, vine/cat snakes, keelbacks) are `non_venomous` by the
   *medically-dangerous-to-humans* rule. This is the correct boundary for the asymmetric
   objective and is the data point reviewers should scrutinize first — see the **flagged**
   species in `species_venom_map.csv`.

3. **The hard-case set is the real metric.** Headline = **recall on venomous**, but the
   number that should gate any release is venomous recall **on the hard-case split**
   (rat-vs-cobra, wolf-vs-krait, distant/low-res). A model can look great on the random
   test split and still fail the confusions that kill — report both, always.

**Recommended next steps (architect to sequence)**
1. Have a herpetologist sign off on the flagged species in `species_venom_map.csv`.
2. Baseline = binary venom classifier on the deduped train split; evaluate on `test` **and**
   `hardcase`, reporting venomous recall + confusion on the mimic pairs.
3. Treat the GitHub set as **train-only / non-releasable** (scraped, license-unclear);
   prioritize licensed venomous + look-alike images to lift hard-case recall.